In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR = '/content/drive/My Drive/covid_xray_data'
# Output: accuracy/loss curves and confusion matrices
FIG_OUTPUT_DIR = '/content/drive/My Drive/covid_results'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_OUTPUT_DIR, exist_ok=True)
%cd $DATA_DIR

def save_fig(name):
    """Save current figure to FIG_OUTPUT_DIR"""
    import matplotlib.pyplot as plt
    d = FIG_OUTPUT_DIR if 'FIG_OUTPUT_DIR' in dir() else './results'
    os.makedirs(d, exist_ok=True)
    plt.savefig(os.path.join(d, name), bbox_inches='tight', dpi=150)
    print('Saved:', os.path.join(d, name))

print('Data dir:', DATA_DIR)
print('Figures will be saved to:', FIG_OUTPUT_DIR)
if not os.path.isdir(os.path.join(DATA_DIR, 'Train')) or not os.path.isdir(os.path.join(DATA_DIR, 'Validation')):
    print('\n*** Upload your dataset: put Train/, Validation/, and Test/ folders inside', DATA_DIR)
    print('    (In Drive: My Drive → covid_xray_data → upload or create Train, Validation, Test)')

In [ ]:
ls

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.autograd import Variable
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy
import cv2
import matplotlib.image as mpimg
import glob
from PIL import Image
from sklearn.metrics import f1_score

In [ ]:
try:
    data_dir = DATA_DIR
except NameError:
    data_dir = 'data'

In [ ]:
#Define transforms for the training data and testing data
train_transforms = transforms.Compose([transforms.RandomRotation(30),
                                       transforms.Resize(256),
                                       transforms.RandomResizedCrop(224),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor(),
                                       transforms.Normalize([0.485, 0.456, 0.406],
                                                            [0.229, 0.224, 0.225])])

test_transforms = transforms.Compose([transforms.Resize(256),
                                      transforms.CenterCrop(224),
                                      transforms.ToTensor(),
                                      transforms.Normalize([0.485, 0.456, 0.406],
                                                           [0.229, 0.224, 0.225])])

#pass transform here-in
train_data = datasets.ImageFolder(root='Train', transform=train_transforms)
# test_data = datasets.ImageFolder(root= 'test', transform=test_transforms)
valid_data = datasets.ImageFolder(root= 'Validation', transform=test_transforms)

print(len(valid_data))
print(len(train_data))
# print(len(test_data))

#data loaders
trainloader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True,num_workers=4)
# testloader = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=True,num_workers=4)
validloader = torch.utils.data.DataLoader(valid_data, batch_size=64, shuffle=True,num_workers=4)

print(len(trainloader))

print("Classes: ")
class_names = train_data.classes
print(class_names)

In [ ]:
def imshow(inp, title=None):
    inp = inp.numpy().transpose((1, 2, 0))
    plt.axis('off')
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)

def show_databatch(inputs, classes):
    out = torchvision.utils.make_grid(inputs)
    imshow(out, title=[class_names[x] for x in classes])

# Get a batch of training data
inputs, classes = next(iter(trainloader))
show_databatch(inputs, classes)